# Full Reaction Network

\begin{align}
CO_2 + 3H_2 &\rightleftharpoons CH_3OH + H_2O\\
CO + 2H_2 &\rightleftharpoons CH_3OH\\
CO_2 + H_2 &\rightleftharpoons CO + H_2O\\
2CH_3OH &\rightleftharpoons CH_3OCH_3 + H_2O
\end{align}


# Yield

"How much of Educt i is converted into Product k"

\begin{equation}
    Y_{k} = \frac{\nu_i}{\nu_k} \, \frac{n_{k, 0}-n_{k}}{n_{i,0}}
\end{equation}

Educt is Methanol, everything else is a byproduct

# Reaction extend

\begin{equation}
    \xi_{i} = \frac{n_i - n_{i,0}}{\nu_i}
\end{equation}

# Importing the researched Glenn Data-Sets

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as scp
import pandas as pd

# Pfad zu den Daten
data_path = r'c:\Users\Peter\Documents\Uni\Master\CRE 3\CRE3-Assignment-2\data_literature\NASA_shomate_coefficients_full.csv'

# NASA Glenn Koeffizienten laden
df_nasa = pd.read_csv(data_path)
print("\nGeladene Spezies:")
print(df_nasa[['species_name', 'formula', 'T1_min_K', 'T1_max_K', 'T2_min_K', 'T2_max_K']].to_string())


Geladene Spezies:
  species_name  formula  T1_min_K  T1_max_K  T2_min_K  T2_max_K
0           H2       H2     200.0    1000.0    1000.0    6000.0
1          H2O      H2O     200.0    1000.0    1000.0    6000.0
2           CO       CO     200.0    1000.0    1000.0    6000.0
3          CO2      CO2     200.0    1000.0    1000.0    6000.0
4        CH3OH    CH3OH     200.0    1000.0    1000.0    6000.0
5      CH3OCH3  CH3OCH3     200.0    1000.0    1000.0    6000.0


# Formulating the Matrix of stoichometric coefficients

\begin{equation}
  \underline{N}^T=
    \begin{bmatrix}
    & CO & CO_2 & CH_3OCH_3 & H_2 & H_2O & CH_3OH \\
R_1 & 0 & -1 & 0 & -3 & 1 & 1 \\
R_2 & -1 & 0 & 0 & -2 & 0 & 1 \\
R_3 & 1 & -1 & 0 & -1 & 1 & 0 \\
R_4 & 0 & 0 & 1 & 0 & 1 & -2
    \end{bmatrix}
\end{equation}

To ensure, that the mathematical method of the matrix of stochiometric coefficients is possible, we need to snure, that $N_{1,1}$ has an inverse matrix that can be calculated. A quick calculation shows, that R3 is equal to R1 -R2 ($R_1 - R_2 = R_3$). 

Therefor, R4 and R3 are interchanged, to enable a succesfull solution of the mathematical approach:

\begin{equation}
  \underline{N}^T=
    \begin{bmatrix}
    & CO & CO_2 & CH_3OCH_3 & H_2 & H_2O & CH_3OH \\
R_1 & 0 & -1 & 0 & -3 & 1 & 1 \\
R_2 & -1 & 0 & 0 & -2 & 0 & 1 \\
R_4 & 0 & 0 & 1 & 0 & 1 & -2  \\
R_3 & 1 & -1 & 0 & -1 & 1 & 0 
    \end{bmatrix}
\end{equation}

In [2]:
import numpy as np
from numpy.linalg import matrix_rank

# --- DME Synthesis Stoichiometric Analysis ---
# Component Order: [CO, CO2, CH3OCH3 (DME), H2, H2O, CH3OH]
# Reaction Order adjusted: [R1, R2, R4, R3] to ensure N11 is non-singular
# This order ensures Key Components are in the first 3 rows (Rank = 3)

n_matrix = np.array([
    [ 0, -1,  0,  1],  # CO    
    [-1,  0,  0, -1],  # CO2   
    [ 0,  0,  1,  0],  # CH3OCH3 (DME) -> Now independent in the 3rd column (because R1-R2=R3)
    [-3, -2,  0, -1],  # H2    
    [ 1,  0,  1,  1],  # H2O   
    [ 1,  1, -2,  0]   # CH3OH
])

# Get the transposed matrix (N^T)
n_transposed = n_matrix.T

# Determine the rank of the matrix
n_rank = matrix_rank(n_matrix)

print("--- DME Synthesis Stoichiometric Analysis ---")
print(f"Sequence: [CO, CO2, CH3OCH3 (DME), H2, H2O, CH3OH]")
print("\nTransposed Matrix (N^T):")
print(n_transposed)
print(f"\nMatrix Rank (Number of Key Reactions): {n_rank}")

# creating a list of species and their corresponding indexes in the transposed matrix/reactions 1-4
# Index 0 = R1, Index 1 = R2, Index 2 = R4, Index 3 = R3 (adjusted order for independence)
species = ['CO', 'CO2', 'CH3OCH3', 'H2', 'H2O', 'CH3OH']
reaction_tuples = np.full((len(n_transposed), len(species)), '', dtype=object)

for i in range(4):
    for j in range(len(species)):
        reaction_tuples[i][j] = (species[j], n_transposed[i][j])

--- DME Synthesis Stoichiometric Analysis ---
Sequence: [CO, CO2, CH3OCH3 (DME), H2, H2O, CH3OH]

Transposed Matrix (N^T):
[[ 0 -1  0 -3  1  1]
 [-1  0  0 -2  0  1]
 [ 0  0  1  0  1 -2]
 [ 1 -1  0 -1  1  0]]

Matrix Rank (Number of Key Reactions): 3


The rank is $R_N=3$, which means that three key reactions and components are sufficient to describe the reaction extent based on stoichiometry. According to the order of components and reactions chosen for the matrix of stoichiometric coefficients, $CO$, $CO_2$, $CH_3OCH_3 (DME)$ are the key components. 

## 2. Thermodynamische Funktionen (Glenn-Gleichung NASA-Format)

NASA-Glenn Koeffizienten (7 pro Bereich):
$$\frac{C_p^\circ(T)}{R} = a_1 T^{-2} + a_2 T^{-1} + a_3 + a_4 T + a_5 T^2 + a_6 T^3 + a_7 T^4$$

$$\frac{H^\circ(T)}{RT} = -a_1 T^{-2} + a_2 \frac{\ln T}{T} + a_3 + \frac{a_4}{2} T + \frac{a_5}{3} T^2 + \frac{a_6}{4} T^3 + \frac{a_7}{5} T^4 + \frac{b_1}{T}$$

$$\frac{S^\circ(T)}{R} = -\frac{a_1}{2} T^{-2} - a_2 T^{-1} + a_3 \ln T + a_4 T + \frac{a_5}{2} T^2 + \frac{a_6}{3} T^3 + \frac{a_7}{4} T^4 + b_2$$


In [3]:
# Storing Glenn-coefficients in Dictionary
GLENN_NASA = {}

for _, row in df_nasa.iterrows():
    species = row['species_name'].strip()
    # Checks, if the given row has only one or two temperature ranges. If T2_min_K is NaN, there is only one range.
    if pd.isna(row['T2_min_K']): 
        # Only one Temperature range
        ranges = [(
            row['T1_min_K'], row['T1_max_K'],
            row['a1_T1'], row['a2_T1'], row['a3_T1'], row['a4_T1'], row['a5_T1'],
            row['a6_T1'], row['a7_T1'], row['b1_T1'], row['b2_T1']
        )]
    else:
        # Two temperature ranges
        ranges = [
            (
                row['T1_min_K'], row['T1_max_K'],
                row['a1_T1'], row['a2_T1'], row['a3_T1'], row['a4_T1'], row['a5_T1'],
                row['a6_T1'], row['a7_T1'], row['b1_T1'], row['b2_T1']
            )
        ]
        # Additional check to ensure that the second range is valid before appending
        if not pd.isna(row['T2_min_K']):
            ranges.append((
                row['T2_min_K'], row['T2_max_K'],
                row['a1_T2'], row['a2_T2'], row['a3_T2'], row['a4_T2'], row['a5_T2'],
                row['a6_T2'], row['a7_T2'], row['b1_T2'], row['b2_T2']
            ))
    GLENN_NASA[species] = ranges

# getting the enthalpy of formation at 298 K for a given species:
def get_hf298(species):
    dict_species = df_nasa.set_index('species_name')
    return dict_species.loc[species]['hf298_J_mol']


# for sp, ranges in GLENN_NASA.items():
#     print(f"  {sp}: {len(ranges)} range(s)")
#     for r in ranges:
#         print(f"    {r[0]:.0f} – {r[1]:.0f} K")

## Coding the NASA-Glenn-equations

In [4]:
R = 8.314  # J/(mol*K)

def cp_T(species, T):
    """
    Calculate the specific heat capacity (Cp) at a given temperature T for a specified species.
    
    Parameters:
    - species: The name of the species (string).
    - T: Temperature in Kelvin (float).
    
    Returns:
    - Cp: Specific heat capacity in J/(mol*K) (float).
    """
    if species not in GLENN_NASA:
        raise ValueError(f"Species '{species}' not found in GLENN_NASA data.")
    
    # Find the correct temperature range for the given species
    for r in GLENN_NASA[species]:
        T_min, T_max = r[0], r[1]
        if T_min <= T <= T_max:
            a1, a2, a3, a4, a5, a6, a7, b1, b2 = r[2], r[3], r[4], r[5], r[6], r[7], r[8], r[9], r[10]
            # Calculate Cp using the NASA polynomial coefficients
            Cp = R * (a1 * T**(-2) + a2 * T**(-1) + a3 + a4 * T + a5 * T**2 + a6 * T**3 + a7 * T**4)
            return Cp
    
    raise ValueError(f"Temperature {T} K is out of range for species '{species}'.")

def hf_T(species, T):
    """
    Calculate the enthalpy of formation (hf) at a given temperature T for a specified species.
    
    Parameters:
    - species: The name of the species (string).
    - T: Temperature in Kelvin (float).
    
    Returns:
    - hf: Enthalpy of formation in J/mol (float).
    """
    if species not in GLENN_NASA:
        raise ValueError(f"Species '{species}' not found in GLENN_NASA data.")
    
    # Get the enthalpy of formation at 298 K
    # hf_298 = get_hf298(species)
    
    # Find the correct temperature range for the given species
    for r in GLENN_NASA[species]:
        T_min, T_max = r[0], r[1]
        if T_min <= T <= T_max:
            a1, a2, a3, a4, a5, a6, a7, b1, b2 = r[2], r[3], r[4], r[5], r[6], r[7], r[8], r[9], r[10]
            # Calculate the change in enthalpy from 298 K to T using the NASA polynomial coefficients
            delta_hf = R * T * ((-a1 * T**(-2)) + a2 * (np.log(T)/T) + a3 + (a4/2) * T + (a5/3) * T**2 + (a6/4) * T**3 + (a7/5) * T**4 + b1/T )
            # hf = delta_hf + hf_298
            return delta_hf
    
    raise ValueError(f"Temperature {T} K is out of range for species '{species}'.")

def s_T(species, T):
    """
    Calculate the entropy (S) at a given temperature T for a specified species.
    
    Parameters:
    - species: The name of the species (string).
    - T: Temperature in Kelvin (float).
    
    Returns:
    - S: Entropy in J/(mol*K) (float).
    """
    if species not in GLENN_NASA:
        raise ValueError(f"Species '{species}' not found in GLENN_NASA data.")
    
    # Find the correct temperature range for the given species
    for r in GLENN_NASA[species]:
        T_min, T_max = r[0], r[1]
        if T_min <= T <= T_max:
            a1, a2, a3, a4, a5, a6, a7, b1, b2 = r[2], r[3], r[4], r[5], r[6], r[7], r[8], r[9], r[10]
            # Calculate S using the NASA polynomial coefficients
            S = R * (-a1 * T**(-2) - a2 * T**(-1) + a3 * np.log(T) + a4 * T + (a5/2) * T**2 + (a6/3) * T**3 + (a7/4) * T**4 + b2)
            return S
    
    raise ValueError(f"Temperature {T} K is out of range for species '{species}'.")


## Calculating Reaction-enthalpy, -entropy and -gibbs energy

In [5]:
def reaction_enthalpy(reaction, T):
    """
    Calculate the reaction enthalpy (ΔH) at a given temperature T for a specified reaction.
    
    Parameters:
    - reaction: A list of tuples representing the reaction. Each tuple contains (species, coefficient).
                Coefficients should be negative for reactants and positive for products.
    - T: Temperature in Kelvin (float).
    
    Returns:
    - ΔH: Reaction enthalpy in J/mol (float).
    """
    delta_H = 0.0
    for species, coeff in reaction:
        hf = hf_T(species, T)
        delta_H += coeff * hf
    return delta_H

def reaction_entropy(reaction, T):
    """
    Calculate the reaction entropy (ΔS) at a given temperature T for a specified reaction.
    
    Parameters:
    - reaction: A list of tuples representing the reaction. Each tuple contains (species, coefficient).
                Coefficients should be negative for reactants and positive for products.
    - T: Temperature in Kelvin (float).
    
    Returns:
    - ΔS: Reaction entropy in J/(mol*K) (float).
    """
    delta_S = 0.0
    for species, coeff in reaction:
        S = s_T(species, T)
        delta_S += coeff * S
    return delta_S

def reaction_gibbs_free_energy(reaction, T):
    """
    Calculate the reaction Gibbs free energy (ΔG) at a given temperature T for a specified reaction.
    
    Parameters:
    - reaction: A list of tuples representing the reaction. Each tuple contains (species, coefficient).
                Coefficients should be negative for reactants and positive for products.
    - T: Temperature in Kelvin (float).
    
    Returns:
    - ΔG: Reaction Gibbs free energy in J/mol (float).
    """
    delta_H = reaction_enthalpy(reaction, T)
    delta_S = reaction_entropy(reaction, T)
    delta_G = delta_H - T * delta_S
    return delta_G

def eq_const_p(reaction, T):
    """
    Calculate the equilibrium constant (K) at a given temperature T for a specified reaction.
    
    Parameters:
    - reaction: A list of tuples representing the reaction. Each tuple contains (species, coefficient).
                Coefficients should be negative for reactants and positive for products.
    - T: Temperature in Kelvin (float).
    
    Returns:
    - K: Equilibrium constant (dimensionless).
    """
    delta_G = reaction_gibbs_free_energy(reaction, T)
    K = np.exp(-delta_G / (R * T))
    return K

# k_x = K_p * p^Δν

def stoich_coeff_change(reaction):
    """Change in molar stoichiometric coefficient Δν = Σ vi."""
    dnu = 0
    for species, coeff in reaction:
        dnu += coeff
    return dnu

def eq_const_x(reaction, T, p):
    """
    Calculate the equilibrium constant (K_x) at a given temperature T and pressure p for a specified reaction.
    
    Parameters:
    - reaction: A list of tuples representing the reaction. Each tuple contains (species, coefficient).
                Coefficients should be negative for reactants and positive for products.
    - T: Temperature in Kelvin (float).
    - p: Pressure in bar (float).
    
    Returns:
    - K_x: Equilibrium constant (dimensionless).
    """
    K_p = eq_const_p(reaction, T)
    dnu = stoich_coeff_change(reaction)
    K_x = K_p * (p ** (-dnu))
    return K_x

print('Gibbs')
print('R1:', reaction_gibbs_free_energy(reaction_tuples[0], 500))
print('R2:', reaction_gibbs_free_energy(reaction_tuples[1], 500))
print('R4:', reaction_gibbs_free_energy(reaction_tuples[2], 500))
print('R3:', reaction_gibbs_free_energy(reaction_tuples[3], 500))
print('----------------------------------')
print('Enthalpy')
print('R1:', reaction_enthalpy(reaction_tuples[0], 500))
print('R2:', reaction_enthalpy(reaction_tuples[1], 500))
print('R4:', reaction_enthalpy(reaction_tuples[2], 500))
print('R3:', reaction_enthalpy(reaction_tuples[3], 500))
print('----------------------------------')
print('Entropy')
print('R1:', reaction_entropy(reaction_tuples[0], 500))
print('R2:', reaction_entropy(reaction_tuples[1], 500))
print('R4:', reaction_entropy(reaction_tuples[2], 500))
print('R3:', reaction_entropy(reaction_tuples[3], 500))

Gibbs
R1: 38085.197022887514
R2: 18590.215714246966
R4: -11518.253975078022
R3: 19494.981308640585
----------------------------------
Enthalpy
R1: -57858.376139356755
R2: -97670.84843482183
R4: -21955.403286191693
R3: 39812.47229546512
----------------------------------
Entropy
R1: -191.88714632448853
R2: -232.5221282981376
R4: -20.874298622227343
R3: 40.634981973649076


## Getting $\xi$ from $K_x$

\begin{align}
    K_x &= \prod_i x_i^{\nu_i}
\end{align}

$K_x$ is known from calculations above, we need to solve for the equillibrium molar fractions of all species. This is done, by changing the formular to '0' on one side, and root solving the equation. E.g. for R1:

\begin{align}
    0 &= K_x \cdot x_{CO_2} \cdot x_{H_2}^3 - x_{CH_{3}OH} \cdot x_{H_{2}O}
\end{align}

with 

\begin{align}
  x_{i,out} &= \frac{\dot n_{i,out}}{\dot n_{out}}=\frac{\dot n_{i,in} + \nu_i\,\xi}{\dot n_{in} + \xi\,\sum_i \nu_i}
\end{align}

By guessing $\xi$ with initial values and then iteratively solving for the root, we get values for $\xi$

In [59]:
from scipy.optimize import root

def material_balance(reaction, xi, n_in):
    """
    Calculate the material balance for a given reaction, extent of reaction (xi), and initial moles (n_in).
    
    Parameters:
    - reaction: A list of tuples representing the reaction. Each tuple contains (species, coefficient).
                Coefficients should be negative for reactants and positive for products.
    - xi: Extent of reactions (float).
    - n_in: Initial moles of each species as an array [mol/h].
    
    Returns:
    - n_out: Final moles of each species after the reaction as an array [mol/h].
    """
    n_out = n_in.copy()                     # Start with initial moles
    for j in range(len(reaction)):
        species, coeff = reaction[j]
        n_out[j] = n_out[j] + (coeff * xi[0])
    return n_out

def mole_fraction(n_out):
    """
    Calculate the mole fraction of each species given the final moles.
    
    Parameters:
    - n_out: Final moles of each species as an array [mol/h].
    
    Returns:
    - x: Mole fraction of each species as an array.
    """
    total_moles = np.sum(n_out)
    if total_moles == 0:
        return np.zeros_like(n_out)  # Avoid division by zero
    x = n_out / total_moles
    return x

def root_solver(reaction, xi_init, n_in, T, p, reac_index):
    """
    Wrapper function for root solver to find the extent of reaction (xi) that satisfies the equilibrium condition.
    
    Parameters:
    - reaction: A list of tuples representing the reaction. Each tuple contains (species, coefficient).
                Coefficients should be negative for reactants and positive for products.
    - xi_init: Initial guess for the extent of reaction (float).
    - n_in: Initial moles of each species as an array [mol/h]. MUST have the same order as the species in the reaction dictionary for all reactions R1-R4. MUST be initialized as a float array, else the function will not work properly.
    - T: Temperature in Kelvin (float).
    - p: Pressure in bar (float).
    - reac_index: Index of the reaction for which to solve (int). R1 = 1, R2 = 2, R3 = 3, R4 = 4.
    Returns:
    - xi_solution: Extent of reaction that satisfies the equilibrium condition (float).
    """
    def equilibrium_condition(xi):
        n_out = material_balance(reaction, xi, n_in)
        x = mole_fraction(n_out)
        K_x = eq_const_x(reaction, T, p)
        stoich_coeff_storage = []
        for i in range(len(reaction)):
            species, coeff = reaction[i]
            stoich_coeff_storage.append(float(coeff))
        
        if reac_index == 1:
            res = K_x * (x[1]**stoich_coeff_storage[1]) * (x[3]**stoich_coeff_storage[3]) - (x[5]**stoich_coeff_storage[5]) * (x[4]**stoich_coeff_storage[4])
        elif reac_index == 2:
            res = K_x * (x[0]**stoich_coeff_storage[0]) * (x[3]**stoich_coeff_storage[3]) - (x[5]**stoich_coeff_storage[5])
        elif reac_index == 3:
            res = K_x * (x[1]**stoich_coeff_storage[1]) * (x[3]**stoich_coeff_storage[3]) - (x[0]**stoich_coeff_storage[0]) * (x[4]**stoich_coeff_storage[4])
        elif reac_index == 4:
            res = K_x * (x[5]**stoich_coeff_storage[5]) - (x[2]**stoich_coeff_storage[2]) * (x[4]**stoich_coeff_storage[4])
        
        return float(res)
    solution = root(equilibrium_condition, xi_init)
    return solution.x


def root_solver_wrapper(reaction_list, xi_init, n_in, T, p, reac_index):
    """
    Wrapper function for root solver to find the extent of reaction (xi) that satisfies the equilibrium condition for a list of reactions.
    
    Parameters:
    - reaction_list: A list of reactions, where each reaction is a list of tuples representing the reaction. Each tuple contains (species, coefficient). Coefficients should be negative for reactants and positive for products.
    - xi_init: Initial guess for the extent of reaction (array).
    - n_in: Initial moles of each species as an array [mol/h]. MUST have the same order as the species in the reaction dictionary for all reactions R1-R4.
    - T: Temperature in Kelvin (float).
    - p: Pressure in bar (float).
    - reac_index: Index of the reaction for which to solve (int). R1 = 1, R2 = 2, R3 = 3, R4 = 4.
    
    Returns:
    - xi_solutions: A list of extents of reaction that satisfy the equilibrium condition for each reaction in the list.
    """
    xi_solutions = []
    for i in range(len(reaction_list)):
        xi_solution = root_solver(reaction_list[i], xi_init, n_in, T, p, reac_index)
        xi_solutions.append(xi_solution)
    return xi_solutions


# print(reaction_tuples[0])

# print('Root Solver Test for R1:', root_solver(reaction_tuples[0], [0.1], np.array([1.0, 1.0, 0.0, 2.0, 0.0, 0.0]), 500, 1, 1))
# print('Root Solver Test for R2:', root_solver(reaction_tuples[1], [0.1], np.array([1, 1, 0, 2, 0, 0]), 500, 1, 2))
# print('Root Solver Test for R3:', root_solver(reaction_tuples[2], [0.1], np.array([1, 1, 0, 2, 0, 0]), 500, 1, 3))
# print('Root Solver Test for R4:', root_solver(reaction_tuples[3], [0.1], np.array([1, 1, 0, 2, 0, 0]), 500, 1, 4))